In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys, gc, random, logging
from collections import Counter

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1,3,4,5,6")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append("../..")

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from rich.console import Console
from rich.theme import Theme
from rich.markup import escape

from src.models import ModelandTokenizer
from src.tokens import prepare_input
from src.functional import get_hs
from src.utils import env_utils, logging_utils
from src.goals.multiline_prompts import (
    build_example,
    replace_in_code_newline_with_tabnewline,
    VAR_LETTERS,
)


logging_utils.configure(level=logging.INFO)
console = Console(theme=Theme({"default": "white"}), highlight=False)


def rich_print(*a, **k):
    console.print(*a, **k)


logger = logging.getLogger(__name__)

# MODEL_NAME = "Qwen/Qwen2.5-Coder-14B-Instruct"
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"

# regen vs cache: True -> always rebuild pairs + re-extract latents;  False -> load from local-disk cache when present
OVERWRITE = False

In [3]:
mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}, n_embd={mt.n_embd}")

2026-06-19 14:16:19 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


2026-06-19 14:16:19 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-19 14:16:19 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-06-19 14:16:19 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-19 14:16:19 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-19 14:16:19 httpx INFO     HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Coder-32B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-06-19 14:16:19 httpx

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-06-19 14:16:40 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-19 14:16:40 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-06-19 14:16:40 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64, n_embd=5120


In [4]:
# ── local-disk cache for pairs + latents (gated by OVERWRITE from the setup cell) ──
import json, pickle, hashlib

CACHE_DIR = os.path.join(env_utils.DEFAULT_RESULTS_DIR, "multiline_signal",
                         MODEL_NAME.split("/")[-1], "cache")
os.makedirs(CACHE_DIR, exist_ok=True)


def _key(**cfg):
    """Short stable hash of every config field that changes the cached data."""
    return hashlib.md5(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:10]


def cache_path(kind, ext, **cfg):
    return os.path.join(CACHE_DIR, f"{kind}_{cfg.get('mode', '')}_{_key(**cfg)}.{ext}")


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)


print(f"cache dir: {CACHE_DIR}\nOVERWRITE = {OVERWRITE}")

cache dir: /disk/u/arnab/Codes/Projects/goals/results/multiline_signal/Qwen2.5-Coder-32B-Instruct/cache
OVERWRITE = False


# 04.3 · Learning a **stop** / **continue** signal (mass-mean-diff)

Reuses the forced-prefill prompt setup from `04.2_multiline.ipynb` (`src.goals.multiline_prompts`).

**Idea.** At the pivot (the trailing whitespace right after the target condition's prefill) the LM
is either:
- **continue** — it was given the target's *incomplete* first line (`multiline` style) and should
  keep evaluating the same variable, or
- **stop / move-on** — it was given a *complete* one-liner (`singleline` style) and should move to
  the next variable.

We look for a single linear direction that captures this *intent*, invariant to the concrete
predicate. Following the TODO:

- **v_stop** = `E[h_singleline − h_multiline]`  (points toward "stop / move on")
- **v_continue** = `−v_stop`

`h_*` is the residual-stream latent at a fixed token position (`-1` / `-2`) under each prefill.

**This notebook (mass-mean-diff, the simple thing first):** build minimal contrastive pairs →
extract latents (batched) → check separability with PCA + a linear probe → form `v_stop` /
`v_continue`. The causal steering test is scaffolded at the end (TODO).

## 1. Contrastive pairs (singleline ↔ multiline)

Each pair shares **one per-pair seed**, so both members get identical predicates / params / priors
and differ *only* at the target condition's prefill style. That makes them true **minimal pairs**:
the latent difference isolates the stop-vs-continue intent, not the surrounding content.

In [5]:
# ── build (or load) minimal contrastive pairs: SAME conditions, target rendered two ways ──
SEED        = 0
N_PER_SLOT  = 1000                # pairs per target slot
SLOTS       = [1, 2, 3]          # target = b, c, d  (priors < k are complete, >=1 multiline)
PROMPT_MODE = "chat"             # "chat" | "fim"
POS         = -2                 # latent token position (-1 / -2: trailing whitespace at the pivot)

pairs_cfg = dict(model=MODEL_NAME, seed=SEED, n_per_slot=N_PER_SLOT, slots=SLOTS, mode=PROMPT_MODE)
pairs_fp  = cache_path("pairs", "pkl", **pairs_cfg)

if not OVERWRITE and os.path.exists(pairs_fp):
    pairs = load_pickle(pairs_fp)
    print(f"[cache] loaded {len(pairs)} pairs <- {os.path.basename(pairs_fp)}")
else:
    base_rng = random.Random(SEED)
    pairs = []
    for k in SLOTS:
        for _ in range(N_PER_SLOT):
            seed = base_rng.randrange(2 ** 31)
            ml = build_example(mt, random.Random(seed), target_slot=k, n_conditions=k + 2,
                               mode=PROMPT_MODE, style="multiline")    # incomplete -> continue
            sl = build_example(mt, random.Random(seed), target_slot=k, n_conditions=k + 2,
                               mode=PROMPT_MODE, style="singleline")   # complete   -> stop / move-on
            # tokenization-stable prompt (used for extraction); pad trailing whitespace at the pivot
            ml["prompt"] = replace_in_code_newline_with_tabnewline(ml["full_prompt"]) + "   "
            sl["prompt"] = replace_in_code_newline_with_tabnewline(sl["full_prompt"]) + "   "
            pairs.append({"slot": k, "ml": ml, "sl": sl})
    save_pickle(pairs, pairs_fp)
    print(f"[gen] built {len(pairs)} pairs -> {os.path.basename(pairs_fp)}")

print(f"[{PROMPT_MODE}] {len(pairs)} contrastive pairs"
      f"  | slots: {dict(Counter(p['slot'] for p in pairs))}")

[gen] built 3000 pairs -> pairs_chat_66ebdcb98f.pkl
[chat] 3000 contrastive pairs  | slots: {1: 1000, 2: 1000, 3: 1000}


In [6]:
# sanity: members should share the token at POS (so v_stop reflects upstream intent, not the
# surface token). Report the match rate; show one pair's two target lines.
def tok_at(prompt, pos):
    ids = prepare_input(prompt, tokenizer=mt, padding_side="left")["input_ids"][0]
    return mt.tokenizer.decode(ids[pos].item())

match = sum(tok_at(p["sl"]["prompt"], POS) == tok_at(p["ml"]["prompt"], POS) for p in pairs)
print(f"POS={POS} token-aligned pairs: {match}/{len(pairs)}")

e = pairs[0]
rich_print(f"[bold]slot[/bold]={e['slot']}  target_var={e['ml']['target_var']}  "
           f"pred={e['ml']['target_pred']}")
rich_print("[green]── multiline (continue): forced INCOMPLETE first line ──[/green]")
rich_print(escape(e["ml"]["target_first_line"]))
rich_print("[red]── singleline (stop): forced COMPLETE one-liner ──[/red]")
rich_print(escape(e["sl"]["target_first_line"]))

POS=-2 token-aligned pairs: 3000/3000


slot=1  target_var=b  pred=reversed_greater

── multiline (continue): forced INCOMPLETE first line ──

b = int(str(abs(n))[::-1])

── singleline (stop): forced COMPLETE one-liner ──

b = int(str(abs(n))[::-1]) > abs(n)

## 2. Extract latents (batched, all residual layers)

One forward pass per batch reads every residual layer at `POS`. Left-padding keeps the negative
`POS` aligned to the same real token across the batch.

In [7]:
from tqdm.auto import tqdm

# ── batched latent extraction over ALL residual layers at POS (cached) ──────
LAYERS = list(range(mt.n_layer))
BATCH  = 16

@torch.no_grad()
def extract_latents(prompts, layers, pos, batch_size=BATCH):
    """Residual-stream latents at one token position. Returns (n_prompts, n_layers, d) float32.
    Relies on left-padding so a negative `pos` aligns to the same real token across the batch."""
    locs = [(mt.layer_name_format.format(L), pos) for L in layers]
    out = np.empty((len(prompts), len(layers), mt.n_embd), dtype=np.float32)
    for b0 in tqdm(range(0, len(prompts), batch_size), desc="Extracting latents"):
        chunk = prompts[b0:b0 + batch_size]
        tok = prepare_input(chunk, tokenizer=mt, padding_side="left")
        hs = get_hs(mt=mt, input=tok, locations=locs, return_dict=True)
        for li, L in enumerate(layers):
            h = hs[(mt.layer_name_format.format(L), pos)]
            if h.ndim == 1:
                h = h.unsqueeze(0)
            out[b0:b0 + len(chunk), li] = h.float().cpu().numpy()
        del hs
        gc.collect()
    return out

lat_cfg = dict(**pairs_cfg, pos=POS, n_layer=mt.n_layer)
lat_fp  = cache_path("latents", "npz", **lat_cfg)

if not OVERWRITE and os.path.exists(lat_fp):
    z = np.load(lat_fp)
    H_sl, H_ml = z["H_sl"], z["H_ml"]
    print(f"[cache] loaded latents {H_sl.shape} <- {os.path.basename(lat_fp)}")
else:
    H_sl = extract_latents([p["sl"]["prompt"] for p in pairs], LAYERS, POS)   # stop class (+)
    H_ml = extract_latents([p["ml"]["prompt"] for p in pairs], LAYERS, POS)   # continue class (-)
    np.savez(lat_fp, H_sl=H_sl, H_ml=H_ml)
    print(f"[gen] extracted latents {H_sl.shape} -> {os.path.basename(lat_fp)}")

print("H_sl", H_sl.shape, "| H_ml", H_ml.shape)

Extracting latents:   0%|          | 0/188 [00:00<?, ?it/s]

Extracting latents:   0%|          | 0/188 [00:00<?, ?it/s]

[gen] extracted latents (3000, 64, 5120) -> latents_chat_5171a94518.npz
H_sl (3000, 64, 5120) | H_ml (3000, 64, 5120)


## 3. Separability across layers (animated projection)

Replicates the contrastive-direction GIF from `03_probe.ipynb`. For every `LAYER_STEP`-th layer we
fit `v_stop` (paired mass-mean-diff) on a **train** split and, on the held-out **test** split, plot:

- **x** — projection onto `v_stop` (the stop ⇄ continue axis), with its ROC-AUC;
- **y** — top PC of the residual after removing `v_stop` (left-over off-axis structure), with its AUC.

A clean signal shows the two classes pulling apart along **x** (AUC → 1) over some band of layers,
with little separation remaining on **y**.

In [8]:
# ── per-layer projection (replicates 03_probe), via src.analysis.algo ───────
# X_AXIS picks what the scatter x-axis shows:
#   "v_stop" -> projection onto v_stop (mass-mean-diff);  y = top residual PC
#   "v_cont" -> projection onto v_cont = -v_stop (mirror); y = top residual PC
#   "PC1"    -> plain 2D PCA of the latents: x = PC1, y = PC2
# AUC(stop) / best_layer are always read from the v_stop signal (ProjectionResult.auc_signal),
# independent of X_AXIS, so the table / best layer / saved signal stay comparable across modes.
from src.analysis.algo import projection_across_layers

X_AXIS       = "PC1"     # ["v_stop", "v_cont", "PC1"]
LAYER_STEP   = 4         # compute only every 4th layer (no need for all)
DO_NORMALIZE = True      # L2-normalize latents before projection (cosine geometry)
TEST_FRAC    = 0.5       # v_stop fit on the TRAIN split; points shown on the TEST split

assert X_AXIS in ("v_stop", "v_cont", "PC1")
viz_layers = list(range(0, mt.n_layer, LAYER_STEP))
n_pairs = len(pairs)
perm = np.random.RandomState(0).permutation(n_pairs)
n_test = int(n_pairs * TEST_FRAC)
te, tr = perm[:n_test], perm[n_test:]

# map the X_AXIS choice onto projection_across_layers kwargs
_axis_kw = {
    "v_stop": dict(axis="direction", sign=1.0,  signal_name="v_stop"),
    "v_cont": dict(axis="direction", sign=-1.0, signal_name="v_cont"),
    "PC1":    dict(axis="pca"),
}[X_AXIS]

proj_results = projection_across_layers(
    H_sl, H_ml, viz_layers,
    train_idx=tr, test_idx=te, normalize=DO_NORMALIZE, **_axis_kw,
)

print(f"{'Layer':>6} {'AUC(stop)':>10} {'AUC(resid)':>11} {'Resid Var%':>10}")
print(f"{'-----':>6} {'---------':>10} {'----------':>11} {'---------':>10}")
for L in viz_layers:
    r = proj_results[L]
    print(f"{L:>6} {r.auc_signal:>10.3f} {r.auc_resid:>11.3f} {r.resid_var:>9.1%}")

best_layer = max(viz_layers, key=lambda L: proj_results[L].auc_signal)
print(f"\nX_AXIS={X_AXIS} | best layer = {best_layer}  "
      f"(AUC stop {proj_results[best_layer].auc_signal:.3f}; test split {n_test} pos + {n_test} neg)")

 Layer  AUC(stop)  AUC(resid) Resid Var%
 -----  ---------  ----------  ---------
     0      0.832       0.421     15.2%
     4      0.918       0.554     29.7%
     8      0.942       0.552     30.0%
    12      0.941       0.546     32.9%
    16      0.991       0.559     25.5%
    20      0.994       0.444     27.4%
    24      0.994       0.435     21.7%
    28      0.994       0.436     19.6%
    32      0.993       0.416     18.0%
    36      0.991       0.563     15.0%
    40      0.993       0.573     13.0%
    44      0.996       0.600     10.4%
    48      1.000       0.539      9.1%
    52      1.000       0.498      8.4%
    56      1.000       0.528     10.2%
    60      1.000       0.478     11.6%

X_AXIS=PC1 | best layer = 52  (AUC stop 1.000; test split 1500 pos + 1500 neg)


In [9]:
# ── animated projection across layers (replicates 03_probe contrastive.gif) ──
from IPython.display import HTML
from src.analysis.vis import animate_projection

FIG_DIR = os.path.join(env_utils.DEFAULT_RESULTS_DIR, "multiline_signal",
                       MODEL_NAME.split("/")[-1], "figures")
gif_fp = os.path.join(FIG_DIR, f"stop_continue_{PROMPT_MODE}_pos{POS}_{X_AXIS}.gif")
model_short = MODEL_NAME.split("/")[-1]

anim = animate_projection(
    proj_results,
    pos_label="singleline (stop)", neg_label="multiline (continue)",
    pos_color="tab:red", neg_color="tab:green",
    title_fn=lambda L: f"Layer {L} | {model_short} | {PROMPT_MODE} · POS={POS} · X_AXIS={X_AXIS} · test set",
    save_path=gif_fp,
)
print("saved gif ->", gif_fp)
HTML(anim.to_jshtml())

2026-06-19 14:32:20 matplotlib.animation INFO     Animation.save using <class 'matplotlib.animation.PillowWriter'>
saved gif -> /disk/u/arnab/Codes/Projects/goals/results/multiline_signal/Qwen2.5-Coder-32B-Instruct/figures/stop_continue_chat_pos-2_PC1.gif
2026-06-19 14:32:23 matplotlib.animation INFO     Animation.save using <class 'matplotlib.animation.HTMLWriter'>


## 4. The signal: `v_stop` / `v_continue`

Mass-mean-diff (paired): `v_stop[L] = mean(h_singleline − h_multiline)`, `v_continue = −v_stop`.
Unit-normalized variants are convenient for steering (scale by a coefficient at apply time).

In [15]:
H_sl.shape

(3000, 64, 5120)

In [16]:

# ── mass-mean-diff signal ───────────────────────────────────────────────────
V_stop = (H_sl - H_ml).mean(axis=0)          # (n_layers, d): paired difference of means
V_cont = -V_stop
norms = np.linalg.norm(V_stop, axis=1)
print(f"v_stop per-layer norm: min {norms.min():.2f}  max {norms.max():.2f}  "
      f"@best(L{best_layer}) {norms[best_layer]:.2f}")

V_stop_unit = V_stop / (norms[:, None] + 1e-8)
V_cont_unit = -V_stop_unit

SAVE = False
if SAVE:
    out_dir = os.path.join(env_utils.DEFAULT_RESULTS_DIR, "multiline_signal",
                           MODEL_NAME.split("/")[-1])
    os.makedirs(out_dir, exist_ok=True)
    fp = os.path.join(out_dir, f"massmean_{PROMPT_MODE}_pos{POS}.npz")
    np.savez(fp, V_stop=V_stop, V_stop_unit=V_stop_unit, layers=np.array(LAYERS),
             auc_stop=np.array([proj_results[L].auc_signal for L in viz_layers]),
             viz_layers=np.array(viz_layers), best_layer=best_layer, pos=POS, n_pairs=len(pairs),
             prompt_mode=PROMPT_MODE)
    print("saved ->", fp)

v_stop per-layer norm: min 1.34  max 397.63  @best(L52) 282.76


In [17]:
from src.functional import logit_lens as logit_lens
from src.functional import patchscope

for layer_idx in range(0, mt.n_layer, 2):
    v = torch.Tensor(V_cont[layer_idx]).to(mt.dtype)
    # ll_pred = logit_lens(
    #     mt = mt,
    #     h = v,
    #     k = 5
    # )
    ll_pred = patchscope(
        mt=mt,
        h=v,
        k=5
    )
    print(f"Layer {layer_idx:>2} :  {[str(pred) for pred in ll_pred]}")

Layer  0 :  ["' ' [220] (p=0.181, logit=13.812)", "' iPhone' [12167] (p=0.059, logit=12.688)", "' guitar' [16986] (p=0.049, logit=12.500)", "' AI' [15235] (p=0.040, logit=12.312)", "' iOS' [16048] (p=0.036, logit=12.188)"]
Layer  2 :  ["' W' [467] (p=0.080, logit=11.875)", "' copy' [2975] (p=0.071, logit=11.750)", "' X' [1599] (p=0.055, logit=11.500)", "' Y' [809] (p=0.055, logit=11.500)", "' python' [10135] (p=0.022, logit=10.562)"]
Layer  4 :  ["' y' [379] (p=0.281, logit=13.750)", "' x' [856] (p=0.133, logit=13.000)", "' python' [10135] (p=0.063, logit=12.250)", "' �' [1683] (p=0.043, logit=11.875)", "' java' [1642] (p=0.020, logit=11.125)"]
Layer  6 :  ["' t' [259] (p=0.781, logit=18.375)", "' ' [220] (p=0.068, logit=15.938)", "' s' [274] (p=0.037, logit=15.312)", "' r' [435] (p=0.034, logit=15.250)", "' z' [1147] (p=0.013, logit=14.312)"]
Layer  8 :  ["' r' [435] (p=0.656, logit=17.250)", "' s' [274] (p=0.101, logit=15.375)", "' z' [1147] (p=0.029, logit=14.125)", "' t' [259] (p=0

## 5. Causal experiment (steering)

The signal above is correlational. The test of intent is causal:

- add `coeff · v_continue[L]` to the residual at layer `L`, position `POS` while generating a
  **singleline (complete)** prompt → expect the LM to **keep evaluating the same var** (flip
  `moved_on → continue`);
- symmetrically add `coeff · v_stop[L]` on a **multiline (incomplete)** prompt → force
  `continue → moved_on`.

Sweep `coeff` (start near `‖v_stop[best_layer]‖`) and report flip rate vs the 04.2 behavioral
verdict, per layer. Steering needs a generation-time hook (baukit `TraceDict` / nnsight
generate-with-patch) that adds the vector at *every* decode step — left as a scaffold below.

In [29]:
from src.goals.multiline_prompts import build_example, replace_in_code_newline_with_tabnewline
from src.functional import generate_with_patch

# style, V_COL = "multiline", V_stop
style, V_COL = "singleline", V_cont

example = build_example(
    mt = mt,
    rng = random.Random(2345),
    target_slot = 1,
    n_conditions = 4,
    mode = PROMPT_MODE,
    style = style,
)
prompt = replace_in_code_newline_with_tabnewline(example["full_prompt"]) + "   "

# prompt = "a, b, c, d, e"
# prompt = "1, 2, 3, 4, 5"
prompt = "5, 5, 5, 5, 5"
tokenized = prepare_input(prompt, tokenizer=mt, padding_side="left")
gen = generate_with_patch(
    mt,
    inputs=tokenized,
    n_gen_per_prompt=1,
    max_new_tokens=32,
    do_sample=False,
    remove_prefix=True,
)[0]
rich_print(f"[white]{escape(prompt)}[/white][sea_green3]{escape(gen)}[/sea_green3]")

5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 

In [32]:
from src.functional import get_hs, PatchSpec

TARGET_LAYERS = list(range(35, 45))
STEER_COEFF = 0.3
src_hs = get_hs(
    mt=mt,
    input=tokenized,
    locations=[(mt.layer_name_format.format(L), POS) for L in TARGET_LAYERS],
    return_dict=True,
)

patches = []
for layer_idx in TARGET_LAYERS:
    src_h = src_hs[(mt.layer_name_format.format(layer_idx), POS)]
    patch_h = torch.Tensor(V_COL[layer_idx]).to(src_h.device)
    patch_h = patch_h * STEER_COEFF * src_h.norm() / (patch_h.norm() + 1e-8)
    patches.append(
        PatchSpec(
            location=(mt.layer_name_format.format(layer_idx), POS),
            patch=patch_h,
            strategy="add",
            on_pass="first",
        )
    )
    patches.append(
        PatchSpec(
            location=(mt.layer_name_format.format(layer_idx), -1),
            patch=patch_h,
            strategy="add",
            on_pass="all",
        )
    )


steered_gen = generate_with_patch(
    mt=mt,
    inputs=tokenized,
    n_gen_per_prompt=1,
    max_new_tokens=32,
    do_sample=False,
    remove_prefix=True,
    patches=patches,
)[0]
rich_print(f"[white]{escape(prompt)}[/white][salmon1]{escape(steered_gen)}[/salmon1]")

5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 